In [ ]:
# ---- n0: CPU data prep for extra seeds of Saner Turhaner's spectrum-structure scorer (no training here) ----
# Kit: sanpier/casmi-spectrum-model-v1 (complete training kit on the mount). Recipe after Vedrin Jeliazkov's
# casmi26-scorer-retrain-r1-1028, which reproduced final_model.pt byte for byte. Only change to the kit: prepare.py
# also accepts the 'normalized_smiles' column that the kit's own data_audit.json records for train.parquet.
import hashlib, json, shutil, subprocess, sys, time
from pathlib import Path
T0 = time.time()
def log(*a): print(*a, flush=True)
def sha(p, chunk=8 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()

try:
    from rdkit import rdBase
except ImportError:
    wheel = sorted(Path('/kaggle/input').rglob('rdkit-2026.3.3-*.whl'))[0]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(wheel.parent), 'rdkit==2026.03.3'], check=True)
    from rdkit import rdBase
assert rdBase.rdkitVersion == '2026.03.3', rdBase.rdkitVersion

hits = sorted(Path('/kaggle/input').rglob('casmi_spectrum_model_v1/train.py'))
assert len(hits) == 1, hits
kit = hits[0].parent
log('KIT', kit, sorted(p.name for p in kit.iterdir()))
pins = json.loads((kit / 'sha256.json').read_text())
for name in ('features.py', 'model.py', 'prepare.py', 'train.py', 'inference.py', 'runtime_test.py', 'config.json'):
    assert sha(kit / name) == pins[name], name
stage = Path('/kaggle/working/kit')
stage.mkdir(parents=True, exist_ok=True)
for p in kit.iterdir():
    if p.suffix in ('.py', '.json', '.md', '.txt'):
        shutil.copy2(p, stage / p.name)
prep = stage / 'prepare.py'
old, new = "'smiles':['smiles','canonical_smiles']", "'smiles':['smiles','canonical_smiles','normalized_smiles']"
text = prep.read_text(); assert text.count(old) == 1; prep.write_text(text.replace(old, new))
for name in ('config.json', 'train.py', 'model.py', 'inference.py', 'features.py'):
    log(f'===== {name}'); log((stage / name).read_text())
for name in ('training_result.json', 'environment.json'):
    if (kit / name).is_file():
        log(f'===== {name}'); log((kit / name).read_text()[:4000])

sys.path.insert(0, str(stage))
import prepare, runtime_test
cfg = json.loads((stage / 'config.json').read_text())
trains = sorted(Path('/kaggle/input').rglob('train.parquet'))
assert len(trains) == 1, trains
cfg.update(train_path=str(trains[0]), work='/kaggle/working/cache', output='/kaggle/working/unused')
runtime_test.self_test(cfg)
log('RUNTIME_TEST PASS', round(time.time() - T0), 's')
audit = prepare.prepare(cfg)
log('PREP_AUDIT', json.dumps(audit))
expected = {'peaks.bin': '37c825a721c4fc3076a59a454687e7c3c65b3b401a8e2b22048884fe6cf5703f',
            'metadata.bin': 'f5d814e9fb90370731edbbbd4d991edc599ecc3db320f5a77c05a854d8e433d3',
            'fingerprints.npy': 'cdd019dc6377afbdd1d295489856413de8bc2828420912f038dbc5a2fb687ce8',
            'molecules.parquet': 'a158239c869f4565971d93d37cae0c88c52677e6622807392222118516c71e09',
            'spectra.parquet': 'a1b74c688f09d47330e8c5a672659ada198af66533f80954103c086d8f4a3fae'}
got = {k: sha(Path(cfg['work']) / k) for k in expected}
log('PREP_SHA', json.dumps(got))
log('PREP_MATCHES_PUBLIC_RETRAIN', {k: got[k] == v for k, v in expected.items()})
for p in sorted(Path(cfg['work']).iterdir()):
    log('CACHE', p.name, p.stat().st_size)
log('N0_DONE', round(time.time() - T0), 's')
